In [1]:
import os

In [2]:
%pwd

'e:\\repo\\CICD\\CI_CD3_DL\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'e:\\repo\\CICD\\CI_CD3_DL'

In [ ]:
from datacalsses import dataclasss
from pathlib import Path


@dataclasss
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: str
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    eval_steps: int
    save_steps: int
    gradient_accumulation_steps: int




In [ ]:
from src.projectDL_1.utils.common import read_yaml, create_directories
from src.projectDL_1.constants import *


In [ ]:
class ConfiguartionManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_Model_trainer_config(self) -> ModelTrainerConfig:
        config=self.config.model_trainer

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_ckpt=config.model_ckpt,
            num_train_epochs=config.num_train_epochs,
            warmup_steps=config.warmup_steps,
            per_device_train_batch_size=config.per_device_train_batch_size,
            weight_decay=config.weight_decay,
            logging_steps=config.logging_steps,
            evaluation_strategy=config.evaluation_strategy,
            eval_steps=config.eval_steps,
            save_steps=config.save_steps,
            gradient_accumulation_steps=config.gradient_accumulation_steps
        )
        return model_trainer_config




In [ ]:
from transformers import TrainingArguments, Trainer, AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_from_disk, load_from_disk, Da
import torch

In [ ]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {device}")

        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

        # loading data
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        trainer_args = TrainingArguments(
            output_dir=self.config.root_dir,
            per_device_train_batch_size=self.config.per_device_train_batch_size,
            num_train_epochs=self.config.num_train_epochs,
            warmup_steps=self.config.warmup_steps,
            weight_decay=self.config.weight_decay,
            logging_steps=self.config.logging_steps,
            evaluation_strategy=self.config.evaluation_strategy,
            eval_steps=self.config.eval_steps,
            save_steps=self.config.save_steps,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps
        )

        trainer = Trainer(
            model = model_pegasus,
            args = trainer_args,
            tokenizer = tokenizer,
            train_dataset = dataset_samsum_pt["test"],
            eval_dataset = dataset_samsum_pt["validation"],
            data_collator = seq2seq_data_collator
        )

        trainer.train()

        # saving the trained model
        model_pegasus.save_pretrained(os.path.join(self.config.root_dir, "pegasus_samsum_model"))
        # Save tokenizer
        tokenizer.save_pretrained(os.path.join(self.config.root_dir, "tokenizer"))


In [ ]:
try:
    config = ConfiguartionManager()
    model_trainer_config = config.get_Model_trainer_config()
    print(model_trainer_config)

    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train()

except Exception as e:
    raise e